In [0]:
from pyspark.sql.functions import col, expr, trim

# Read data
df = spark.read.table("mini_project_catalog.`01_bronze`.invoice")

# Remove duplicates
df = df.dropDuplicates()

# Replace invalid values
df = df.replace("-", None)

# Trim columns
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# Convert timestamp
if "invoice_date" in df.columns:
    df = df.withColumn("invoice_date", expr("to_timestamp(invoice_date)"))

# Convert numeric
if "invoice_amount" in df.columns:
    df = df.withColumn(
        "invoice_amount",
        expr("CAST(try_cast(invoice_amount AS DOUBLE) AS DOUBLE)")
    )

# Handle null payment_mode
df = df.withColumn(
    "payment_mode",
    expr("COALESCE(payment_mode, 'UNKNOWN')")
)

# Filter valid records
df = df.filter(col("invoice_id").isNotNull())
df = df.filter(col("order_id").isNotNull())

# ✅ FIX: overwrite with schema update
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("mini_project_catalog.02_silver.invoice")

In [0]:
%sql
SELECT * FROM mini_project_catalog.02_silver.invoice LIMIT 10;